In [57]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install -q transformers datasets librosa soundfile pandas accelerate nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [59]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Codentry/pronunciation"
DATA_PATH = f"{PROJECT_PATH}/data"
MODEL_PATH = f"{PROJECT_PATH}/model"

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

print("Folders ready.")

Folders ready.


In [60]:
import nltk
nltk.download("cmudict")
from nltk.corpus import cmudict

cmu_dict = cmudict.dict()

[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


In [61]:
from datasets import load_dataset

dataset = load_dataset("librispeech_asr", "clean", split="train.100[:5%]")

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

In [62]:
dataset.save_to_disk(f"{DATA_PATH}/raw_librispeech")
print("Raw dataset saved to Drive.")

Saving the dataset (0/1 shards):   0%|          | 0/1427 [00:00<?, ? examples/s]

Raw dataset saved to Drive.


In [63]:
from datasets import load_from_disk

dataset = load_from_disk(f"{DATA_PATH}/raw_librispeech")
print("Dataset reloaded from Drive.")

Dataset reloaded from Drive.


In [64]:
def text_to_phoneme(text):
    words = text.lower().split()
    phonemes = []

    for w in words:
        if w in cmu_dict:
            phonemes.extend(cmu_dict[w][0])

    return " ".join(phonemes)

dataset = dataset.map(lambda x: {"phoneme": text_to_phoneme(x["text"])})

# Remove empty phoneme samples
dataset = dataset.filter(lambda x: len(x["phoneme"]) > 0)

print("Phoneme mapping done.")

Map:   0%|          | 0/1427 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1427 [00:00<?, ? examples/s]

Phoneme mapping done.


In [65]:
dataset = dataset.train_test_split(test_size=0.1)
dataset["validation"] = dataset.pop("test")

print("Split done.")

Split done.


In [67]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_id = "openai/whisper-base"

processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Important for phoneme training
processor.tokenizer.set_prefix_tokens(language=None, task=None)
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

print("Model ready.")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Model ready.


In [68]:
def prepare(batch):
    speech = batch["audio"]["array"]

    inputs = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )

    batch["input_features"] = inputs.input_features[0]

    labels = processor.tokenizer(
        batch["phoneme"],
        return_tensors="pt",
        padding=False
    )

    batch["labels"] = labels.input_ids[0]

    return batch


dataset = dataset.map(
    prepare,
    remove_columns=dataset["train"].column_names
)

dataset.set_format(type="torch")

print("Dataset ready for training.")

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/143 [00:00<?, ? examples/s]

Dataset ready for training.


In [69]:
from dataclasses import dataclass

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor

    def __call__(self, features):

        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch

In [71]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_PATH,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    remove_unused_columns=False,
)

In [72]:
from transformers import Seq2SeqTrainer

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,5.028861,0.490435
2,0.707234,0.305037
3,0.535662,0.260365
4,0.404697,0.239564
5,0.359650,0.234224


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=405, training_loss=1.114929598349112, metrics={'train_runtime': 377.4474, 'train_samples_per_second': 17.009, 'train_steps_per_second': 1.073, 'total_flos': 4.164011753472e+17, 'train_loss': 1.114929598349112, 'epoch': 5.0})

In [73]:
FINAL_MODEL_PATH = f"{MODEL_PATH}/final_model"

trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

print("Training complete. Model saved to Drive.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Model saved to Drive.
